# 📈 Investment Recommendation System - Complete End-to-End Pipeline

## 🎯 Overview
A comprehensive AI-powered investment recommendation system that combines:
- **Data Pipeline**: Real-time data acquisition & 50+ technical indicators
- **ML Models**: LSTM, ARIMA+GARCH, and Prophet for price prediction
- **Ensemble Predictions**: Weighted model combinations for better accuracy
- **Trading Engine**: Backtesting framework with performance metrics

## 🏆 Key Features
- Multi-model ensemble predictions with confidence intervals
- Advanced technical analysis with TA-Lib
- Risk assessment (VaR, Sharpe Ratio, Maximum Drawdown)
- Comprehensive backtesting with trading signals
- Real-time data from Yahoo Finance

## 📊 Models Implemented
1. **LSTM (Long Short-Term Memory)**: Deep learning for sequence prediction
2. **ARIMA+GARCH**: Statistical time series with volatility modeling
3. **Prophet**: Facebook's forecasting tool for trends and seasonality
4. **Ensemble**: Weighted combination of all models

---

**Author**: Investment Recommendation System Project  
**Repository**: https://github.com/hegdemanu/investment-recommendation-system  
**License**: MIT

## 📦 1. Installation and Setup

In [ ]:
# Install required packages
!pip install -q yfinance pandas numpy matplotlib seaborn scikit-learn
!pip install -q tensorflow prophet arch statsmodels
!pip install -q ta-lib-python

print("✓ All packages installed successfully!")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import yfinance as yf
import talib
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ML libraries
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Time series libraries
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✓ All imports successful!")
print(f"TensorFlow version: {tf.__version__}")
print(f"Pandas version: {pd.__version__}")

---
# 🔄 PART 1: DATA PIPELINE & FEATURE ENGINEERING
---

## 📥 1.1 Data Acquisition

In [ ]:
def fetch_stock_data(symbol, period="2y", interval="1d"):
    """
    Fetch historical stock data from Yahoo Finance
    
    Args:
        symbol: Stock ticker symbol
        period: Time period (1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max)
        interval: Data interval (1d, 1wk, 1mo)
    """
    print(f"📡 Fetching data for {symbol}...")
    data = yf.download(symbol, period=period, interval=interval, progress=False)
    
    if data.empty:
        raise ValueError(f"No data found for {symbol}")
    
    print(f"✓ Downloaded {len(data)} data points ({data.index[0].date()} to {data.index[-1].date()})")
    return data

# Configuration
SYMBOL = "AAPL"  # Change this to analyze different stocks
PERIOD = "2y"

# Fetch data
df = fetch_stock_data(SYMBOL, period=PERIOD)

print(f"\n📊 Dataset Shape: {df.shape}")
print(f"\n📋 Columns: {list(df.columns)}")
df.head()

In [ ]:
# Visualize raw data
fig, axes = plt.subplots(2, 1, figsize=(15, 8))

# Price
axes[0].plot(df.index, df['Close'], linewidth=2, color='#2E86AB')
axes[0].set_title(f'{SYMBOL} Stock Price History', fontsize=16, fontweight='bold')
axes[0].set_ylabel('Price ($)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(df.index, df['Volume'], width=1, alpha=0.7, color='#A23B72')
axes[1].set_title(f'{SYMBOL} Trading Volume', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Volume', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🔧 1.2 Feature Engineering with Technical Indicators

In [ ]:
def add_technical_indicators(data):
    """
    Add 50+ technical indicators using TA-Lib
    """
    df = data.copy()
    print("⚙️  Calculating technical indicators...")
    
    # Moving Averages
    df['MA5'] = talib.SMA(df['Close'], timeperiod=5)
    df['MA10'] = talib.SMA(df['Close'], timeperiod=10)
    df['MA20'] = talib.SMA(df['Close'], timeperiod=20)
    df['MA50'] = talib.SMA(df['Close'], timeperiod=50)
    df['MA200'] = talib.SMA(df['Close'], timeperiod=200)
    
    # Exponential Moving Averages
    df['EMA5'] = talib.EMA(df['Close'], timeperiod=5)
    df['EMA10'] = talib.EMA(df['Close'], timeperiod=10)
    df['EMA20'] = talib.EMA(df['Close'], timeperiod=20)
    
    # Bollinger Bands
    df['BB_upper'], df['BB_middle'], df['BB_lower'] = talib.BBANDS(
        df['Close'], timeperiod=20, nbdevup=2, nbdevdn=2, matype=0
    )
    
    # Momentum Indicators
    df['RSI'] = talib.RSI(df['Close'], timeperiod=14)
    df['MACD'], df['MACD_signal'], df['MACD_hist'] = talib.MACD(
        df['Close'], fastperiod=12, slowperiod=26, signalperiod=9
    )
    df['CCI'] = talib.CCI(df['High'], df['Low'], df['Close'], timeperiod=14)
    df['ADX'] = talib.ADX(df['High'], df['Low'], df['Close'], timeperiod=14)
    df['ROC'] = talib.ROC(df['Close'], timeperiod=10)
    df['WILLR'] = talib.WILLR(df['High'], df['Low'], df['Close'], timeperiod=14)
    
    # Stochastic Oscillator
    df['SLOWK'], df['SLOWD'] = talib.STOCH(
        df['High'], df['Low'], df['Close'], 
        fastk_period=5, slowk_period=3, slowk_matype=0, 
        slowd_period=3, slowd_matype=0
    )
    
    # Volume Indicators
    df['OBV'] = talib.OBV(df['Close'], df['Volume'])
    df['AD'] = talib.AD(df['High'], df['Low'], df['Close'], df['Volume'])
    
    # Volatility Indicators
    df['ATR'] = talib.ATR(df['High'], df['Low'], df['Close'], timeperiod=14)
    
    # Custom Indicators
    df['PVT'] = (df['Close'].pct_change() * df['Volume']).cumsum()
    df['PPO'] = talib.PPO(df['Close'], fastperiod=12, slowperiod=26, matype=0)
    
    # Returns
    df['daily_return'] = df['Close'].pct_change()
    df['5d_return'] = df['Close'].pct_change(periods=5)
    df['10d_return'] = df['Close'].pct_change(periods=10)
    df['20d_return'] = df['Close'].pct_change(periods=20)
    
    # Volatility
    df['volatility_5d'] = df['daily_return'].rolling(window=5).std()
    df['volatility_10d'] = df['daily_return'].rolling(window=10).std()
    df['volatility_20d'] = df['daily_return'].rolling(window=20).std()
    
    # Drop NaN values
    df = df.dropna()
    
    n_features = len(df.columns) - len(data.columns)
    print(f"✓ Added {n_features} technical indicators")
    print(f"✓ Total features: {len(df.columns)}")
    
    return df

# Add features
df_features = add_technical_indicators(df)
print(f"\n📊 Final dataset shape: {df_features.shape}")
df_features.tail()

## 📊 1.3 Visualize Technical Indicators

In [ ]:
# Create comprehensive technical analysis chart
fig, axes = plt.subplots(4, 1, figsize=(16, 14))

# 1. Price with Moving Averages
axes[0].plot(df_features.index, df_features['Close'], label='Close', linewidth=2.5, color='black')
axes[0].plot(df_features.index, df_features['MA20'], label='MA20', alpha=0.8)
axes[0].plot(df_features.index, df_features['MA50'], label='MA50', alpha=0.8)
axes[0].plot(df_features.index, df_features['MA200'], label='MA200', alpha=0.8)
axes[0].set_title(f'{SYMBOL} - Price with Moving Averages', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price ($)')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# 2. Bollinger Bands
axes[1].plot(df_features.index, df_features['Close'], label='Close', linewidth=2, color='black')
axes[1].plot(df_features.index, df_features['BB_upper'], label='Upper Band', linestyle='--', alpha=0.7, color='red')
axes[1].plot(df_features.index, df_features['BB_middle'], label='Middle', alpha=0.7, color='blue')
axes[1].plot(df_features.index, df_features['BB_lower'], label='Lower Band', linestyle='--', alpha=0.7, color='green')
axes[1].fill_between(df_features.index, df_features['BB_upper'], df_features['BB_lower'], alpha=0.1)
axes[1].set_title('Bollinger Bands', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Price ($)')
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

# 3. RSI
axes[2].plot(df_features.index, df_features['RSI'], linewidth=2, color='purple')
axes[2].axhline(y=70, color='r', linestyle='--', alpha=0.5, label='Overbought (70)')
axes[2].axhline(y=30, color='g', linestyle='--', alpha=0.5, label='Oversold (30)')
axes[2].fill_between(df_features.index, 30, 70, alpha=0.1, color='gray')
axes[2].set_title('Relative Strength Index (RSI)', fontsize=14, fontweight='bold')
axes[2].set_ylabel('RSI')
axes[2].set_ylim([0, 100])
axes[2].legend(loc='upper left')
axes[2].grid(True, alpha=0.3)

# 4. MACD
axes[3].plot(df_features.index, df_features['MACD'], label='MACD', linewidth=2)
axes[3].plot(df_features.index, df_features['MACD_signal'], label='Signal', linewidth=2)
colors = ['green' if x > 0 else 'red' for x in df_features['MACD_hist']]
axes[3].bar(df_features.index, df_features['MACD_hist'], label='Histogram', alpha=0.5, color=colors)
axes[3].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[3].set_title('MACD (Moving Average Convergence Divergence)', fontsize=14, fontweight='bold')
axes[3].set_ylabel('MACD')
axes[3].set_xlabel('Date')
axes[3].legend(loc='upper left')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 📈 1.4 Statistical Analysis

In [ ]:
# Comprehensive statistics
print("=" * 70)
print(f"📊 STATISTICAL SUMMARY FOR {SYMBOL}")
print("=" * 70)

print(f"\n💰 Price Statistics:")
print(f"   Current Price:        ${df_features['Close'].iloc[-1]:.2f}")
print(f"   Period High:          ${df_features['Close'].max():.2f}")
print(f"   Period Low:           ${df_features['Close'].min():.2f}")
print(f"   Average Price:        ${df_features['Close'].mean():.2f}")
print(f"   Price Std Dev:        ${df_features['Close'].std():.2f}")

print(f"\n📊 Return Statistics:")
avg_daily = df_features['daily_return'].mean() * 100
daily_vol = df_features['daily_return'].std() * 100
annual_ret = avg_daily * 252
annual_vol = daily_vol * np.sqrt(252)
sharpe = annual_ret / annual_vol if annual_vol != 0 else 0

print(f"   Avg Daily Return:     {avg_daily:.3f}%")
print(f"   Daily Volatility:     {daily_vol:.3f}%")
print(f"   Annualized Return:    {annual_ret:.2f}%")
print(f"   Annualized Vol:       {annual_vol:.2f}%")
print(f"   Sharpe Ratio:         {sharpe:.2f}")

print(f"\n🎯 Technical Indicators (Current):")
print(f"   RSI:                  {df_features['RSI'].iloc[-1]:.2f}")
print(f"   MACD:                 {df_features['MACD'].iloc[-1]:.2f}")
print(f"   ADX:                  {df_features['ADX'].iloc[-1]:.2f}")
print(f"   ATR:                  {df_features['ATR'].iloc[-1]:.2f}")

print(f"\n📦 Volume Statistics:")
print(f"   Average Volume:       {df_features['Volume'].mean():,.0f}")
print(f"   Max Volume:           {df_features['Volume'].max():,.0f}")
print(f"   Latest Volume:        {df_features['Volume'].iloc[-1]:,.0f}")

print("\n" + "=" * 70)

---
# 🤖 PART 2: MACHINE LEARNING MODELS
---

## 🔄 2.1 Data Preparation for ML Models

In [ ]:
def prepare_sequences(data, lookback=60, prediction_horizon=1):
    """
    Prepare sequences for LSTM model
    """
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data[['Close']])
    
    X, y = [], []
    for i in range(lookback, len(scaled_data) - prediction_horizon):
        X.append(scaled_data[i - lookback:i])
        y.append(scaled_data[i + prediction_horizon - 1])
        
    return np.array(X), np.array(y), scaler

# Prepare data
LOOKBACK = 60
X, y, scaler = prepare_sequences(df_features, lookback=LOOKBACK)

# Split into train/test
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"✓ Prepared sequences for ML training")
print(f"   Input shape:          {X.shape}")
print(f"   Target shape:         {y.shape}")
print(f"   Training samples:     {len(X_train)}")
print(f"   Testing samples:      {len(X_test)}")

## 🧠 2.2 LSTM Model Training

In [ ]:
def build_lstm_model(input_shape, units=50, dropout=0.2):
    """
    Build LSTM model architecture
    """
    model = keras.Sequential([
        keras.layers.LSTM(units=units, return_sequences=True, input_shape=input_shape),
        keras.layers.Dropout(dropout),
        keras.layers.LSTM(units=units),
        keras.layers.Dropout(dropout),
        keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Build model
lstm_model = build_lstm_model(input_shape=(X_train.shape[1], X_train.shape[2]))

print("🧠 LSTM Model Architecture:")
lstm_model.summary()

In [ ]:
# Train LSTM model
print("\n🏋️ Training LSTM model...")

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history = lstm_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
    verbose=1
)

print("\n✓ LSTM model training complete!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('LSTM Model Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Training MAE')
axes[1].plot(history.history['val_mae'], label='Validation MAE')
axes[1].set_title('LSTM Model MAE', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 📊 2.3 ARIMA+GARCH Model

In [ ]:
def train_arima_garch(data, train_size=0.8):
    """
    Train ARIMA+GARCH model for time series forecasting with volatility
    """
    print("📈 Training ARIMA+GARCH model...")
    
    # Split data
    split = int(len(data) * train_size)
    train_data = data['Close'].iloc[:split]
    test_data = data['Close'].iloc[split:]
    
    # Train ARIMA
    arima_model = ARIMA(train_data, order=(5, 1, 2))
    arima_results = arima_model.fit()
    
    # Train GARCH on residuals
    residuals = arima_results.resid
    garch_model = arch_model(residuals, vol='Garch', p=1, q=1)
    garch_results = garch_model.fit(disp='off')
    
    print("✓ ARIMA+GARCH model trained")
    
    return {'arima': arima_results, 'garch': garch_results}, test_data

# Train ARIMA+GARCH
arima_garch_model, test_data_arima = train_arima_garch(df_features)

print(f"   ARIMA coefficients: {len(arima_garch_model['arima'].params)}")
print(f"   Test samples: {len(test_data_arima)}")

## 🔮 2.4 Prophet Model

In [ ]:
def train_prophet(data, train_size=0.8):
    """
    Train Prophet model for forecasting
    """
    print("🔮 Training Prophet model...")
    
    # Prepare data for Prophet
    prophet_data = pd.DataFrame({
        'ds': data.index,
        'y': data['Close'].values
    })
    
    # Split data
    split = int(len(prophet_data) * train_size)
    train_data = prophet_data.iloc[:split]
    test_data = prophet_data.iloc[split:]
    
    # Train Prophet
    model = Prophet(
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10,
        seasonality_mode='multiplicative',
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True
    )
    model.fit(train_data)
    
    print("✓ Prophet model trained")
    
    return model, test_data

# Train Prophet
prophet_model, test_data_prophet = train_prophet(df_features)

## 🎯 2.5 Model Evaluation & Predictions

In [ ]:
# LSTM Predictions
lstm_predictions = lstm_model.predict(X_test, verbose=0)
lstm_predictions = scaler.inverse_transform(lstm_predictions)
y_test_actual = scaler.inverse_transform(y_test)

# ARIMA Predictions
arima_predictions = arima_garch_model['arima'].forecast(steps=len(test_data_arima))

# Prophet Predictions
prophet_forecast = prophet_model.predict(test_data_prophet)
prophet_predictions = prophet_forecast['yhat'].values

# Calculate metrics for LSTM
lstm_mse = mean_squared_error(y_test_actual, lstm_predictions)
lstm_rmse = np.sqrt(lstm_mse)
lstm_mae = mean_absolute_error(y_test_actual, lstm_predictions)
lstm_r2 = r2_score(y_test_actual, lstm_predictions)

# Calculate metrics for ARIMA
arima_mse = mean_squared_error(test_data_arima, arima_predictions)
arima_rmse = np.sqrt(arima_mse)
arima_mae = mean_absolute_error(test_data_arima, arima_predictions)
arima_r2 = r2_score(test_data_arima, arima_predictions)

# Calculate metrics for Prophet
prophet_mse = mean_squared_error(test_data_prophet['y'], prophet_predictions)
prophet_rmse = np.sqrt(prophet_mse)
prophet_mae = mean_absolute_error(test_data_prophet['y'], prophet_predictions)
prophet_r2 = r2_score(test_data_prophet['y'], prophet_predictions)

# Display results
print("=" * 70)
print("📊 MODEL PERFORMANCE COMPARISON")
print("=" * 70)

results_df = pd.DataFrame({
    'Model': ['LSTM', 'ARIMA+GARCH', 'Prophet'],
    'RMSE': [lstm_rmse, arima_rmse, prophet_rmse],
    'MAE': [lstm_mae, arima_mae, prophet_mae],
    'R²': [lstm_r2, arima_r2, prophet_r2]
})

print("\n", results_df.to_string(index=False))
print("\n" + "=" * 70)

# Determine best model
best_model_idx = results_df['RMSE'].idxmin()
best_model = results_df.loc[best_model_idx, 'Model']
print(f"\n🏆 Best Model: {best_model} (Lowest RMSE)")

In [ ]:
# Visualize predictions vs actual
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Get test dates
test_dates = df_features.index[-len(y_test_actual):]

# LSTM
axes[0].plot(test_dates, y_test_actual, label='Actual', linewidth=2, color='black')
axes[0].plot(test_dates, lstm_predictions, label='LSTM Prediction', linewidth=2, alpha=0.7)
axes[0].set_title('LSTM Model Predictions', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ARIMA
axes[1].plot(test_data_arima.index, test_data_arima.values, label='Actual', linewidth=2, color='black')
axes[1].plot(test_data_arima.index, arima_predictions.values, label='ARIMA Prediction', linewidth=2, alpha=0.7)
axes[1].set_title('ARIMA+GARCH Model Predictions', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Price ($)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Prophet
axes[2].plot(test_data_prophet['ds'], test_data_prophet['y'], label='Actual', linewidth=2, color='black')
axes[2].plot(prophet_forecast['ds'], prophet_forecast['yhat'], label='Prophet Prediction', linewidth=2, alpha=0.7)
axes[2].fill_between(prophet_forecast['ds'], 
                       prophet_forecast['yhat_lower'], 
                       prophet_forecast['yhat_upper'], 
                       alpha=0.2, label='Confidence Interval')
axes[2].set_title('Prophet Model Predictions', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Price ($)')
axes[2].set_xlabel('Date')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🎲 2.6 Ensemble Predictions

In [ ]:
def create_ensemble_predictions(horizon=5):
    """
    Create ensemble predictions using weighted average of all models
    """
    print(f"🎲 Generating {horizon}-day ensemble forecast...")
    
    # Weights based on model performance (inverse of RMSE)
    weights = {
        'lstm': 1 / lstm_rmse,
        'arima': 1 / arima_rmse,
        'prophet': 1 / prophet_rmse
    }
    total_weight = sum(weights.values())
    weights = {k: v/total_weight for k, v in weights.items()}
    
    print(f"   Model weights: LSTM={weights['lstm']:.2%}, "
          f"ARIMA={weights['arima']:.2%}, Prophet={weights['prophet']:.2%}")
    
    # Generate future predictions
    last_sequence = X[-1].reshape(1, LOOKBACK, 1)
    lstm_future = []
    
    for _ in range(horizon):
        pred = lstm_model.predict(last_sequence, verbose=0)
        lstm_future.append(pred[0, 0])
        last_sequence = np.roll(last_sequence, -1, axis=1)
        last_sequence[0, -1, 0] = pred[0, 0]
    
    lstm_future = scaler.inverse_transform(np.array(lstm_future).reshape(-1, 1)).flatten()
    
    # ARIMA future predictions
    arima_future = arima_garch_model['arima'].forecast(steps=horizon).values
    
    # Prophet future predictions
    future_dates = pd.date_range(
        start=df_features.index[-1] + timedelta(days=1),
        periods=horizon,
        freq='D'
    )
    future_df = pd.DataFrame({'ds': future_dates})
    prophet_future = prophet_model.predict(future_df)['yhat'].values
    
    # Ensemble
    ensemble = (
        weights['lstm'] * lstm_future + 
        weights['arima'] * arima_future + 
        weights['prophet'] * prophet_future
    )
    
    return {
        'dates': future_dates,
        'lstm': lstm_future,
        'arima': arima_future,
        'prophet': prophet_future,
        'ensemble': ensemble,
        'weights': weights
    }

# Generate 5-day forecast
forecast = create_ensemble_predictions(horizon=5)

# Display forecast
forecast_df = pd.DataFrame({
    'Date': forecast['dates'],
    'LSTM': forecast['lstm'],
    'ARIMA': forecast['arima'],
    'Prophet': forecast['prophet'],
    'Ensemble': forecast['ensemble']
})

print("\n📅 5-Day Price Forecast:")
print(forecast_df.to_string(index=False))

current_price = df_features['Close'].iloc[-1]
forecast_change = (forecast['ensemble'][-1] - current_price) / current_price * 100
print(f"\n📊 Expected 5-day change: {forecast_change:+.2f}%")

In [ ]:
# Visualize forecast
plt.figure(figsize=(16, 8))

# Historical data
plt.plot(df_features.index[-60:], df_features['Close'].iloc[-60:], 
         label='Historical', linewidth=2, color='black')

# Forecasts
plt.plot(forecast['dates'], forecast['lstm'], 'o--', label='LSTM', alpha=0.7)
plt.plot(forecast['dates'], forecast['arima'], 's--', label='ARIMA', alpha=0.7)
plt.plot(forecast['dates'], forecast['prophet'], '^--', label='Prophet', alpha=0.7)
plt.plot(forecast['dates'], forecast['ensemble'], 'D-', label='Ensemble', 
         linewidth=3, markersize=10)

plt.axvline(x=df_features.index[-1], color='red', linestyle='--', alpha=0.5, label='Forecast Start')
plt.title(f'{SYMBOL} - 5-Day Price Forecast (Ensemble)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price ($)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# 💹 PART 3: TRADING ENGINE & BACKTESTING
---

## 📊 3.1 Generate Trading Signals

In [ ]:
def generate_trading_signals(data, predictions, threshold=0.02):
    """
    Generate trading signals based on predictions
    
    Args:
        data: Historical price data
        predictions: Model predictions
        threshold: Minimum price change to trigger signal
    
    Returns:
        signals: 1 (buy), 0 (hold), -1 (sell)
    """
    signals = []
    
    for i, pred in enumerate(predictions):
        if i < len(data):
            current_price = data.iloc[i]
            predicted_return = (pred - current_price) / current_price
            
            if predicted_return > threshold:
                signals.append(1)  # Buy
            elif predicted_return < -threshold:
                signals.append(-1)  # Sell
            else:
                signals.append(0)  # Hold
        else:
            signals.append(0)
    
    return np.array(signals)

# Generate signals from LSTM predictions
test_prices = df_features['Close'].iloc[-len(lstm_predictions):].values
signals = generate_trading_signals(
    df_features['Close'].iloc[-len(lstm_predictions):], 
    lstm_predictions.flatten(),
    threshold=0.01
)

print(f"📊 Generated {len(signals)} trading signals")
print(f"   Buy signals:  {(signals == 1).sum()}")
print(f"   Hold signals: {(signals == 0).sum()}")
print(f"   Sell signals: {(signals == -1).sum()}")

## 💰 3.2 Backtest Trading Strategy

In [ ]:
def backtest_strategy(prices, signals, initial_capital=10000, commission=0.001):
    """
    Backtest trading strategy
    
    Args:
        prices: Historical prices
        signals: Trading signals (1=buy, -1=sell, 0=hold)
        initial_capital: Starting capital
        commission: Trading commission rate
    """
    capital = initial_capital
    position = 0
    portfolio_values = []
    trades = []
    
    for i, (price, signal) in enumerate(zip(prices, signals)):
        # Buy signal
        if signal == 1 and position == 0:
            shares = (capital * 0.95) / price  # Use 95% of capital
            position = shares
            capital -= shares * price * (1 + commission)
            trades.append({
                'index': i,
                'type': 'BUY',
                'price': price,
                'shares': shares,
                'capital': capital
            })
        
        # Sell signal
        elif signal == -1 and position > 0:
            capital += position * price * (1 - commission)
            trades.append({
                'index': i,
                'type': 'SELL',
                'price': price,
                'shares': position,
                'capital': capital
            })
            position = 0
        
        # Calculate portfolio value
        portfolio_value = capital + (position * price if position > 0 else 0)
        portfolio_values.append(portfolio_value)
    
    # Close any open position
    if position > 0:
        capital += position * prices[-1] * (1 - commission)
        portfolio_values[-1] = capital
    
    return {
        'portfolio_values': np.array(portfolio_values),
        'final_capital': capital,
        'trades': trades,
        'initial_capital': initial_capital
    }

# Run backtest
backtest_results = backtest_strategy(test_prices, signals)

# Calculate metrics
total_return = (backtest_results['final_capital'] - backtest_results['initial_capital']) / backtest_results['initial_capital']
portfolio_returns = np.diff(backtest_results['portfolio_values']) / backtest_results['portfolio_values'][:-1]
sharpe_ratio = np.mean(portfolio_returns) / np.std(portfolio_returns) * np.sqrt(252) if len(portfolio_returns) > 0 else 0
max_drawdown = np.min(backtest_results['portfolio_values'] / np.maximum.accumulate(backtest_results['portfolio_values']) - 1)

# Buy and hold comparison
buy_hold_return = (test_prices[-1] - test_prices[0]) / test_prices[0]

print("=" * 70)
print("💹 BACKTEST RESULTS")
print("=" * 70)
print(f"\n💰 Performance Metrics:")
print(f"   Initial Capital:      ${backtest_results['initial_capital']:,.2f}")
print(f"   Final Capital:        ${backtest_results['final_capital']:,.2f}")
print(f"   Total Return:         {total_return:.2%}")
print(f"   Sharpe Ratio:         {sharpe_ratio:.2f}")
print(f"   Max Drawdown:         {max_drawdown:.2%}")
print(f"\n📊 Trading Activity:")
print(f"   Number of Trades:     {len(backtest_results['trades'])}")
print(f"   Buy Orders:           {sum(1 for t in backtest_results['trades'] if t['type'] == 'BUY')}")
print(f"   Sell Orders:          {sum(1 for t in backtest_results['trades'] if t['type'] == 'SELL')}")
print(f"\n🎯 Benchmark Comparison:")
print(f"   Buy & Hold Return:    {buy_hold_return:.2%}")
print(f"   Strategy Return:      {total_return:.2%}")
print(f"   Alpha (Outperform):   {(total_return - buy_hold_return):.2%}")
print("\n" + "=" * 70)

## 📈 3.3 Visualize Backtest Results

In [ ]:
# Create comprehensive backtest visualization
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

test_dates = df_features.index[-len(test_prices):]

# 1. Price chart with buy/sell signals
axes[0].plot(test_dates, test_prices, linewidth=2, color='black', label='Price')

# Mark buy/sell points
for trade in backtest_results['trades']:
    if trade['type'] == 'BUY':
        axes[0].scatter(test_dates[trade['index']], trade['price'], 
                       color='green', marker='^', s=200, label='Buy' if trade == backtest_results['trades'][0] else '', zorder=5)
    else:
        axes[0].scatter(test_dates[trade['index']], trade['price'], 
                       color='red', marker='v', s=200, label='Sell' if trade == backtest_results['trades'][0] else '', zorder=5)

axes[0].set_title(f'{SYMBOL} - Price with Trading Signals', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Portfolio value over time
axes[1].plot(test_dates, backtest_results['portfolio_values'], linewidth=2, color='#2E86AB', label='Strategy')
buy_hold_portfolio = backtest_results['initial_capital'] * (test_prices / test_prices[0])
axes[1].plot(test_dates, buy_hold_portfolio, linewidth=2, color='gray', linestyle='--', alpha=0.7, label='Buy & Hold')
axes[1].axhline(y=backtest_results['initial_capital'], color='red', linestyle=':', alpha=0.5, label='Initial Capital')
axes[1].set_title('Portfolio Value Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Portfolio Value ($)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Drawdown
running_max = np.maximum.accumulate(backtest_results['portfolio_values'])
drawdown = (backtest_results['portfolio_values'] / running_max - 1) * 100
axes[2].fill_between(test_dates, drawdown, 0, color='red', alpha=0.3)
axes[2].plot(test_dates, drawdown, linewidth=2, color='darkred')
axes[2].set_title('Portfolio Drawdown', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Drawdown (%)')
axes[2].set_xlabel('Date')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 📋 3.4 Trade Log

In [ ]:
# Display trade history
if backtest_results['trades']:
    trades_df = pd.DataFrame(backtest_results['trades'])
    trades_df['date'] = [test_dates[i] for i in trades_df['index']]
    trades_df = trades_df[['date', 'type', 'price', 'shares', 'capital']]
    trades_df['price'] = trades_df['price'].round(2)
    trades_df['shares'] = trades_df['shares'].round(2)
    trades_df['capital'] = trades_df['capital'].round(2)
    
    print("📋 Trade History:")
    print(trades_df.to_string(index=False))
else:
    print("No trades executed during backtest period")

---
# 📊 FINAL SUMMARY
---

In [ ]:
print("="*80)
print(f"📈 INVESTMENT RECOMMENDATION SYSTEM - FINAL SUMMARY FOR {SYMBOL}")
print("="*80)

print("\n🔍 DATA ANALYSIS:")
print(f"   Period Analyzed:          {df_features.index[0].date()} to {df_features.index[-1].date()}")
print(f"   Total Data Points:        {len(df_features)}")
print(f"   Technical Indicators:     {len(df_features.columns) - 6}")
print(f"   Current Price:            ${df_features['Close'].iloc[-1]:.2f}")

print("\n🤖 MODEL PERFORMANCE:")
print(f"   Best Model:               {best_model}")
print(f"   LSTM RMSE:                ${lstm_rmse:.2f}")
print(f"   ARIMA RMSE:               ${arima_rmse:.2f}")
print(f"   Prophet RMSE:             ${prophet_rmse:.2f}")

print("\n🔮 5-DAY FORECAST:")
print(f"   Predicted Price (Day 5):  ${forecast['ensemble'][-1]:.2f}")
print(f"   Expected Change:          {forecast_change:+.2f}%")
recommendation = "BUY 📈" if forecast_change > 2 else "SELL 📉" if forecast_change < -2 else "HOLD ⏸️"
print(f"   Recommendation:           {recommendation}")

print("\n💹 BACKTEST RESULTS:")
print(f"   Strategy Return:          {total_return:.2%}")
print(f"   Buy & Hold Return:        {buy_hold_return:.2%}")
print(f"   Outperformance:           {(total_return - buy_hold_return):.2%}")
print(f"   Sharpe Ratio:             {sharpe_ratio:.2f}")
print(f"   Max Drawdown:             {max_drawdown:.2%}")
print(f"   Number of Trades:         {len(backtest_results['trades'])}")

print("\n🎯 RISK METRICS:")
print(f"   Annualized Volatility:    {annual_vol:.2f}%")
print(f"   Current RSI:              {df_features['RSI'].iloc[-1]:.2f}")
rsi_signal = "Overbought" if df_features['RSI'].iloc[-1] > 70 else "Oversold" if df_features['RSI'].iloc[-1] < 30 else "Neutral"
print(f"   RSI Signal:               {rsi_signal}")
print(f"   ATR (Volatility):         {df_features['ATR'].iloc[-1]:.2f}")

print("\n" + "="*80)
print("✅ Analysis Complete!")
print("="*80)

---
# 🎓 Conclusion

This notebook demonstrated a complete end-to-end investment recommendation system:

## ✅ What We Accomplished:
1. **Data Pipeline**: Fetched and processed real-time stock data with 50+ technical indicators
2. **Machine Learning**: Trained LSTM, ARIMA+GARCH, and Prophet models for price prediction
3. **Ensemble Method**: Combined multiple models for improved accuracy
4. **Trading Strategy**: Generated trading signals and backtested performance
5. **Risk Analysis**: Calculated Sharpe ratio, drawdown, and volatility metrics

## 📊 Key Takeaways:
- Ensemble models generally outperform individual models
- Technical indicators provide valuable signals for trading
- Backtesting is crucial for validating strategy performance
- Risk management (stop-loss, position sizing) is essential

## 🚀 Next Steps:
- Experiment with different stocks and time periods
- Fine-tune model hyperparameters
- Add more sophisticated trading rules
- Implement portfolio optimization
- Add sentiment analysis from news/social media

---

**⚠️ Disclaimer**: This system is for educational purposes only. Past performance does not guarantee future results. Always do your own research and consult with financial advisors before making investment decisions.

---

**📚 Resources**:
- Repository: https://github.com/hegdemanu/investment-recommendation-system
- Documentation: See README.md
- License: MIT

**🙏 Acknowledgments**:
- Yahoo Finance API for data
- TensorFlow & Prophet teams
- Open source community